**Estudio de la fuente: precios mayoristas de ODEPA, 2026**

Antes de publicar un solo precio conviene saber si el archivo que entrega ODEPA viene limpio. Este cuaderno recorre el archivo mayorista de 2026 con un procedimiento ordenado, paso a paso, y muestra lo que fuimos encontrando en el camino. En cada paso contamos qué vamos a revisar y por qué, dejamos el código a la vista y explicamos qué apareció. Cuando algo no cuadra, lo registramos como anomalía, junto con la decisión que tomamos y la forma en que el proyecto lo resuelve. Las cifras de los textos corresponden al archivo publicado hasta el 30 de octubre de 2026.

**Carga del archivo**

Para trabajar necesitamos cuatro librerías: `requests` para descargar el archivo, `pandas` para manejarlo como tabla, e `io` y `re` para leerlo en memoria y buscar patrones en el texto.

In [1]:
# io: leer lo descargado en memoria; re: expresiones regulares; pandas: tablas; requests: descarga.
import io
import re

import pandas as pd
import requests

Descargamos el archivo directamente desde el portal de datos de ODEPA, sin guardarlo en el disco, y lo dejamos en una tabla llamada `raw`. Leemos todos los valores como texto, tal como vienen, para que nada se corrija solo sin que lo veamos.

In [2]:
URL = "https://datos.odepa.gob.cl/dataset/33f10516-acbe-4446-b633-68244b9b6b26/resource/580beca0-e87e-4dd4-9e8a-0bd92773f4a6/download/precio_mayorista_fruta-hortaliza_2026.csv"

# La entidad certificadora de ODEPA (emSign) salió de la lista de certifi; por eso se indica de forma explícita.
response = requests.get(URL, timeout=300, verify="../agent/certs/emsign_root_ca_g1.pem")
response.raise_for_status()
# dtype=str conserva cada valor tal como lo escribió la fuente.
raw = pd.read_csv(io.BytesIO(response.content), dtype=str, encoding="utf-8-sig")

print(f"Se cargaron {len(raw):,} filas y {raw.shape[1]} columnas del archivo mayorista 2026.".replace(",", "."))

Se cargaron 155.857 filas y 14 columnas del archivo mayorista 2026.


**Cómo lo vamos a revisar**

Con 155.857 filas no tiene sentido leer el archivo de arriba abajo. Lo que hacemos es hacerle preguntas, de lo más general a lo más particular, y dejar que el código cuente cuántas filas responden de una u otra forma. Son ocho pasos:

| Paso | Pregunta | Qué se busca |
|---|---|---|
| 1. Estructura | ¿Qué trae el archivo y en qué forma? | columnas, tipos y una muestra de filas |
| 2. Completitud | ¿Faltan datos? | vacíos, días y mercados sin filas |
| 3. Validez | ¿Los valores tienen sentido? | rangos y coherencia entre mínimo, promedio y máximo |
| 4. Unicidad | ¿Hay filas repetidas? | filas que coinciden en su clave |
| 5. Categorías | ¿Cuántas formas hay de escribir cada cosa? | unidades, calidades, variedades y orígenes |
| 6. Escala | ¿Un mismo producto tiene precios de escalas muy distintas? | precio por producto y unidad |
| 7. Atípicos | ¿Hay precios fuera de lo habitual? | cada precio contra la mediana de su grupo |
| 8. Comparabilidad | ¿Los envases cambian de una semana a otra? | envases de cada producto por semana |

Dos funciones pequeñas nos ayudan a leer los resultados. `number` escribe las cifras con punto de miles, y `show` presenta las tablas sin los números de fila y con la columna importante en amarillo. No cambian ningún dato, solo cómo se ven.

In [3]:
# La columna de la unidad de venta.
UNIT = "Unidad de comercializacion"


# Escribe un número para leer: miles con punto y dos decimales solo bajo 100.
def number(value):
    if not float(value).is_integer() and abs(value) < 100:
        return f"{value:.2f}".rstrip("0").rstrip(".").replace(".", ",")
    return f"{value:,.0f}".replace(",", ".")


# Tabla para leer: sin números de fila y con las columnas indicadas destacadas.
def show(table, mark=None):
    if isinstance(table, pd.Series):
        table = table.to_frame()
    view = table.style.format(number, subset=table.select_dtypes("number").columns)
    if table.index.name is None:
        view = view.hide(axis="index")
    if mark:
        view = view.set_properties(
            subset=mark, **{"background-color": "rgba(245, 197, 66, 0.3)", "font-weight": "bold"}
        )
    return view

**Paso 1 · ¿Qué trae el archivo?**

Antes de buscar errores hay que saber qué esperar de cada columna, y lo más simple es mirar. Tomamos 100 filas al azar, siempre las mismas, para que cualquiera pueda repetir el ejercicio.

In [4]:
# Muestra al azar con semilla fija, para que se pueda repetir.
print(f"Muestra aleatoria de 100 filas, de un total de {number(len(raw))}.")
raw.sample(100, random_state=7).sort_index()

Muestra aleatoria de 100 filas, de un total de 155.857.


,Fecha,ID region,Region,Mercado,Subsector,Producto,Variedad / Tipo,Calidad,Unidad de comercializacion,Origen,Volumen,Precio minimo,Precio maximo,Precio promedio
1717,2026-01-06,8,Región del Biobío,Vega Monumental Concepción,Frutas,Nectarín,Pulpa amarilla,Segunda,$/bandeja 18 kilos granel,Región de O'Higgins,200,"10000,0000","10000,0000","10000,0000"
1757,2026-01-06,13,Región Metropolitana de Santiago,Vega Central Mapocho de Santiago,Frutas,Palta,Edranol,Segunda,$/kilo (en caja de 17 kilos),Provincia de Quillota,55,"1600,0000","1600,0000","1600,0000"
4415,2026-01-09,13,Región Metropolitana de Santiago,Mercado Mayorista Lo Valledor de Santiago,Hortalizas y tubérculos,Tomate,Semiduro,Extra,$/bandeja 18 kilos,Región de O'Higgins,500,"12000,0000","12000,0000","12000,0000"
5255,2026-01-12,13,Región Metropolitana de Santiago,Vega Central Mapocho de Santiago,Hortalizas y tubérculos,Cebollín,Sin especificar,Segunda,$/paquete 36 unidades,Región Metropolitana,250,"2800,0000","3000,0000","2900,0000"
6853,2026-01-14,13,Región Metropolitana de Santiago,Vega Central Mapocho de Santiago,Hortalizas y tubérculos,Zapallo,Camote,1a nueva(o),$/kilo (volumen en unidades),Región Metropolitana,1240,"400,0000","600,0000","500,0000"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144978,2026-10-09,4,Región de Coquimbo,Terminal La Palmera de La Serena,Frutas,Palta,Edranol,Primera,$/kilo (en caja de 17 kilos),Provincia de Limarí,160,"1600,0000","1700,0000","1650,0000"
145474,2026-10-13,5,Región de Valparaíso,Femacal de La Calera,Frutas,Frutilla,Sin especificar,Primera,$/bandeja 7 kilos,Provincia de Melipilla,68,"8000,0000","8000,0000","8000,0000"
150548,2026-10-22,9,Región de La Araucanía,Vega Modelo de Temuco,Hortalizas y tubérculos,Betarraga,Sin especificar,Primera,$/docena de paquetes,Provincia de Cautín,80,"12000,0000","13000,0000","12500,0000"
155083,2026-10-29,4,Región de Coquimbo,Terminal La Palmera de La Serena,Hortalizas y tubérculos,Zapallo,Camote,1a nueva(o),$/kilo (volumen en unidades),Perú,1300,"2300,0000","2400,0000","2350,0000"


Mirar filas ayuda, pero no alcanza para 155.857. Le pedimos entonces al código un resumen de cada columna: cuántos valores distintos tiene, cómo luce su valor más común (cambiando cada dígito por un 9 y cada palabra por una «a») y qué porcentaje de sus valores parece un número.

In [5]:
# La forma de un valor: cada dígito pasa a 9 y cada tramo de letras, a "a".
def form(column):
    forms = raw[column].str.replace(r"\d", "9", regex=True)
    forms = forms.str.replace(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+", "a", regex=True)
    return forms.value_counts().index[0]


print(f"Resumen de las {raw.shape[1]} columnas: valores distintos, forma más común y porcentaje con forma de número.")
structure = pd.DataFrame(
    {
        "valores distintos": raw.nunique(),
        "forma más común": [form(column) for column in raw.columns],
        "parecen número (%)": [raw[c].str.fullmatch(r"\d+(,\d+)?").mean() * 100 for c in raw.columns],
    }
).rename_axis("columna")
show(structure, mark=["parecen número (%)"])

Resumen de las 14 columnas: valores distintos, forma más común y porcentaje con forma de número.


,valores distintos,forma más común,parecen número (%)
columna,,,
Fecha,209,9999-99-99,0
ID region,9,99,100
Region,9,a a a a,0
Mercado,12,a a a a,0
Subsector,2,a a a,0
Producto,81,a,0
Variedad / Tipo,208,a a,0
Calidad,55,a,0
Unidad de comercializacion,148,$/a 99 a,0


El resumen deja claro que todo llega como texto. Cinco columnas tienen forma de número: el identificador de región, el volumen y los tres precios. Los precios, además, vienen escritos con coma decimal y cuatro decimales, como `4000,0000`.

**Anomalía 1 · Los números llegan como texto.** Precios y volúmenes vienen como texto, así que no se pueden sumar ni comparar tal cual. A primera vista tampoco queda claro si la coma separa decimales o miles. Como todos los precios tienen exactamente cuatro dígitos después de la coma, la leemos como decimal. Convertimos los valores a número y guardamos el texto original al lado, por si alguna vez hay que revisar la conversión.

**Paso 2 · ¿Faltan datos?**

Un dato que falta no siempre se nota a simple vista. Revisamos dos cosas: si hay celdas vacías y si faltan días completos. Empezamos por las celdas.

In [6]:
# Vacíos y espacios al borde, por columna.
stripped = raw.apply(lambda column: column.str.strip())
print(
    f"Se encontraron {number(int(raw.isna().sum().sum() + (stripped == '').sum().sum()))} valores vacíos y "
    f"{number(int((raw != stripped).sum().sum()))} con espacios al borde."
)
show(
    pd.DataFrame(
        {
            "vacíos": raw.isna().sum() + (stripped == "").sum(),
            "espacios al borde": (raw != stripped).sum(),
        }
    ).rename_axis("columna"),
    mark=["vacíos"],
)

Se encontraron 0 valores vacíos y 0 con espacios al borde.


,vacíos,espacios al borde
columna,,
Fecha,0,0
ID region,0,0
Region,0,0
Mercado,0,0
Subsector,0,0
Producto,0,0
Variedad / Tipo,0,0
Calidad,0,0
Unidad de comercializacion,0,0


Un archivo diario no debería saltarse días hábiles. Buscamos los días de lunes a viernes que no tienen ninguna fila y los cruzamos con el calendario oficial de feriados de Chile, porque un día sin datos puede ser un feriado o una falla de publicación.

In [7]:
# Feriados de Chile en días hábiles, entre la primera y la última fecha del archivo (calendario oficial).
HOLIDAYS = {
    "2026-04-03": "Viernes Santo",
    "2026-05-01": "Día del Trabajo",
    "2026-05-21": "Día de las Glorias Navales",
    "2026-06-29": "San Pedro y San Pablo",
    "2026-07-16": "Virgen del Carmen",
    "2026-09-18": "Fiestas Patrias",
    "2026-10-12": "Encuentro de Dos Mundos",
}
weekdays = pd.bdate_range(raw["Fecha"].min(), raw["Fecha"].max()).strftime("%Y-%m-%d")
missing = sorted(set(weekdays) - set(raw["Fecha"]))
print(
    f"Se encontraron {len(missing)} días hábiles sin filas; {len(set(missing) & set(HOLIDAYS))} coinciden con "
    "feriados oficiales."
)
show(pd.DataFrame({"fecha": missing, "feriado": [HOLIDAYS.get(day, "—") for day in missing]}))

Se encontraron 7 días hábiles sin filas; 7 coinciden con feriados oficiales.


fecha,feriado
2026-04-03,Viernes Santo
2026-05-01,Día del Trabajo
2026-05-21,Día de las Glorias Navales
2026-06-29,San Pedro y San Pablo
2026-07-16,Virgen del Carmen
2026-09-18,Fiestas Patrias
2026-10-12,Encuentro de Dos Mundos


Que el archivo tenga datos un día no significa que todos los mercados hayan informado ese día. Contamos cuántos días aparece cada mercado.

In [8]:
# Días con filas de cada mercado, frente a los días del archivo.
days = raw.groupby("Mercado")["Fecha"].nunique()
print("Días con datos de cada mercado, de menor a mayor.")
show(
    pd.DataFrame({"días con filas": days, "% de los días del archivo": days / raw["Fecha"].nunique() * 100})
    .rename_axis("mercado")
    .sort_values("días con filas"),
    mark=["días con filas"],
)

Días con datos de cada mercado, de menor a mayor.


,días con filas,% de los días del archivo
mercado,,
Comercializadora del Agro de Limarí,84,"40,19"
Vega Monumental Concepción,168,"80,38"
Feria Lagunitas de Puerto Montt,194,"92,82"
Terminal La Palmera de La Serena,206,"98,56"
Mapocho venta directa de Santiago,207,"99,04"
Terminal Hortofrutícola Agro Chillán,208,"99,52"
Agrícola del Norte S.A. de Arica,209,100
Femacal de La Calera,209,100
Macroferia Regional de Talca,209,100


No hay celdas vacías ni espacios sobrantes, y los siete días hábiles que faltan son todos feriados. Lo que sí aparece es que algunos mercados informan bastante menos días que el resto.

**Anomalía 2 · Días y mercados sin datos.** Los siete días hábiles sin filas son feriados, como el 1 de mayo y el 18 de septiembre, así que ahí no hay nada que corregir. Con los mercados es distinto: Comercializadora del Agro de Limarí aparece en 84 de los 209 días, y Vega Monumental Concepción, en 168. El archivo no dice si esos días no abrieron o simplemente no informaron. Decidimos no rellenar nada. El precio de cada semana se calcula con los días que existen, y si llega a faltar una semana entera entre otras con datos, la carga se detiene.

**Paso 3 · ¿Los valores tienen sentido?**

Un precio en cero, o un mínimo mayor que el máximo, arruinaría cualquier cálculo posterior. Para buscarlos primero pasamos los precios y el volumen a número, siempre al lado del texto original, y después contamos las filas que no pasan cada control.

In [9]:
# Columnas de trabajo junto al texto original; lo que no es número queda vacío y se cuenta.
for name, column in {
    "low": "Precio minimo",
    "high": "Precio maximo",
    "avg": "Precio promedio",
    "volume": "Volumen",
}.items():
    raw[name] = pd.to_numeric(raw[column].str.replace(",", ".", regex=False), errors="coerce")

numbers = raw[["low", "high", "avg", "volume"]]
checks = {
    "precio o volumen que no es número": numbers.isna().any(axis=1).sum(),
    "precio o volumen en cero o negativo": (numbers <= 0).any(axis=1).sum(),
    "mínimo mayor que máximo": (raw["low"] > raw["high"]).sum(),
    "promedio fuera de mínimo y máximo": ((raw["avg"] < raw["low"]) | (raw["avg"] > raw["high"])).sum(),
    "fecha fuera de 2026": (~raw["Fecha"].str.startswith("2026")).sum(),
}
print(
    f"Se aplicaron {len(checks)} controles a las {number(len(raw))} filas; "
    f"{sum(value > 0 for value in checks.values())} de ellos encontraron filas con problemas."
)
show(pd.Series(checks, name="filas que fallan").rename_axis("control"), mark=["filas que fallan"])

Se aplicaron 5 controles a las 155.857 filas; 0 de ellos encontraron filas con problemas.


,filas que fallan
control,
precio o volumen que no es número,0
precio o volumen en cero o negativo,0
mínimo mayor que máximo,0
promedio fuera de mínimo y máximo,0
fecha fuera de 2026,0


Ningún control falla. No hay valores que no sean número, ni precios en cero o negativos, ni mínimos por encima del máximo. Queda una pregunta más fina: si el promedio publicado es simplemente el punto medio entre el mínimo y el máximo.

In [10]:
# El punto medio entre el mínimo y el máximo de cada fila, contra su promedio publicado.
midpoint = (raw["low"] + raw["high"]) / 2
different = raw[(raw["avg"] - midpoint).abs() >= 0.001]
print(
    f"Se encontraron {number(len(different))} filas cuyo promedio no coincide con el punto medio entre el "
    "mínimo y el máximo."
)
show(different.head(5)[["Producto", "Precio minimo", "Precio maximo", "Precio promedio"]])

Se encontraron 25.600 filas cuyo promedio no coincide con el punto medio entre el mínimo y el máximo.


Producto,Precio minimo,Precio maximo,Precio promedio
Acelga,"17000,0000","18000,0000","17666,6666"
Acelga,"550,0000","600,0000","571,4285"
Ají,"30000,0000","31000,0000","30222,2222"
Ají,"18000,0000","20000,0000","19379,3103"
Ajo,"17000,0000","18000,0000","17428,5714"


**Anomalía 3 · El promedio no es el punto medio.** En 25.600 filas no lo es. Entre $17.000 y $18.000, por ejemplo, el promedio publicado es $17.666 y no $17.500. Ninguna fila queda fuera de su rango, así que no parece un error; puede que ODEPA pondere lo que vendió cada comerciante, aunque no lo documenta. Usamos el promedio tal como viene. ODEPA vio las ventas de ese día y nosotros solo tenemos el mínimo y el máximo, de modo que recalcularlo sería perder información.

In [11]:
# Cuántas veces es el máximo respecto del mínimo, fila por fila.
raw["max_over_min"] = raw["high"] / raw["low"]
wide = raw[raw["max_over_min"] > 2]
print(f"Se encontraron {len(wide)} filas cuyo precio máximo supera el doble del mínimo.")
show(
    wide.sort_values("max_over_min", ascending=False)[
        ["Fecha", "Producto", "Unidad de comercializacion", "Precio minimo", "Precio maximo", "max_over_min"]
    ],
    mark=["max_over_min"],
)

Se encontraron 6 filas cuyo precio máximo supera el doble del mínimo.


Fecha,Producto,Unidad de comercializacion,Precio minimo,Precio maximo,max_over_min
2026-03-11,Zapallo,$/kilo (volumen en unidades),"300,0000","4000,0000","13,33"
2026-07-10,Palta,$/bandeja 10 kilos,"2000,0000","22000,0000",11
2026-09-21,Brócoli,$/unidad,"300,0000","800,0000","2,67"
2026-01-02,Choclo,$/unidad,"100,0000","250,0000","2,5"
2026-04-15,Brócoli,$/unidad,"200,0000","500,0000","2,5"
2026-01-16,Limón,$/bandeja 18 kilos,"7000,0000","15000,0000","2,14"


**Anomalía 4 · Mínimos y máximos muy separados.** Solo seis filas tienen un máximo de más del doble del mínimo, pero algunas llaman la atención: un zapallo entre $300 y $4.000, o una bandeja de palta de 10 kilos entre $2.000 y $22.000, el mismo día y en el mismo mercado. Para el cálculo usamos solo el precio promedio. El mínimo y el máximo nos sirven para descartar lo imposible, y si el promedio se aleja demasiado, lo detecta el control de atípicos del paso 7 (anomalía 16).

**Paso 4 · ¿Hay filas repetidas?**

Una fila duplicada contaría dos veces el mismo precio, pero una que solo parece duplicada no se debe borrar. Buscamos las dos cosas: filas idénticas, y filas que coinciden en día, mercado, producto, variedad, calidad y unidad.

In [12]:
# Las filas que comparten su clave: día, mercado, producto, variedad, calidad y unidad.
key = ["Fecha", "Mercado", "Producto", "Variedad / Tipo", "Calidad", UNIT]
repeated = raw[raw.duplicated(key, keep=False)]
different_price = (repeated.groupby(key)["avg"].nunique() > 1).sum()
print(
    f"Se encontraron {number(raw.duplicated().sum())} filas idénticas y {number(len(repeated))} filas que "
    f"comparten su clave, en {number(repeated.groupby(key).ngroups)} grupos; en {number(different_price)} de ellos "
    "los precios difieren."
)
show(
    repeated.sort_values(key).head(6)[["Fecha", "Mercado", "Producto", UNIT, "Origen", "Volumen", "Precio promedio"]],
    mark=["Origen"],
)

Se encontraron 0 filas idénticas y 18.742 filas que comparten su clave, en 8.948 grupos; en 6.336 de ellos los precios difieren.


Fecha,Mercado,Producto,Unidad de comercializacion,Origen,Volumen,Precio promedio
2026-01-02,Femacal de La Calera,Cebolla,$/malla 18 kilos,Provincia de Santiago,90,"11722,2222"
2026-01-02,Femacal de La Calera,Cebolla,$/malla 18 kilos,Provincia de Melipilla,45,"12000,0000"
2026-01-02,Femacal de La Calera,Choclo,$/unidad,Provincia de Quillota,13000,"200,0000"
2026-01-02,Femacal de La Calera,Choclo,$/unidad,Región Metropolitana,17500,"214,8571"
2026-01-02,Femacal de La Calera,Papa,$/saco 25 kilos,Provincia de Quillota,160,"7000,0000"
2026-01-02,Femacal de La Calera,Papa,$/saco 25 kilos,Región Metropolitana,190,"7000,0000"


No hay ni una fila idéntica. Sí hay muchas que coinciden en todo lo anterior y se diferencian en una sola columna, el origen.

**Anomalía 5 · Filas que parecen repetidas.** Son 18.742 filas, y en 6.336 de sus grupos los precios son distintos. Es el mismo producto, el mismo día y en el mismo mercado, pero traído desde lugares diferentes. No son duplicados, son ventas distintas. Las conservamos todas, y el precio de la semana las pondera según lo que trae cada una.

**Paso 5 · ¿De cuántas formas se escribe cada cosa?**

Las columnas de texto libre son las más traicioneras. Si una misma cosa se escribe de varias maneras, o si una etiqueta junta cosas distintas, los precios dejan de ser comparables. Empezamos contando cuántos valores distintos tiene cada una.

In [13]:
# Valores distintos y valor más frecuente de las columnas de texto libre.
columns = ["Producto", "Variedad / Tipo", "Calidad", UNIT, "Origen"]
print("Valores distintos y valor más frecuente de las columnas de texto libre.")
categories = pd.DataFrame(
    {
        "valores distintos": raw[columns].nunique(),
        "valor más frecuente": [raw[c].value_counts().index[0] for c in columns],
        "% de las filas": [round(raw[c].value_counts(normalize=True).iloc[0] * 100, 1) for c in columns],
    }
).rename_axis("columna")
show(categories, mark=["valores distintos"])

Valores distintos y valor más frecuente de las columnas de texto libre.


,valores distintos,valor más frecuente,% de las filas
columna,,,
Producto,81,Palta,"7,3"
Variedad / Tipo,208,Sin especificar,"36,3"
Calidad,55,Primera,"48,9"
Unidad de comercializacion,148,$/unidad,"8,8"
Origen,59,Región de O'Higgins,18


De las cinco, la que más importa para los precios es la unidad de venta, con 148 formas. Para entender de qué están hechas, separamos el tipo de envase, que es la palabra que viene después de «$/».

In [14]:
# El tipo de envase es la palabra que sigue a «$/».
packaging = raw[UNIT].str.extract(r"^\$/(\w+)")[0]

types = raw.assign(envase=packaging).groupby("envase").agg(formas=(UNIT, "nunique"), filas=(UNIT, "size"))
print(
    f"Se encontraron {types.formas.sum()} formas de escribir la unidad de venta, agrupadas en "
    f"{len(types)} tipos de envase."
)
show(types.sort_values("filas", ascending=False).rename_axis("tipo de envase"), mark=["formas"])

Se encontraron 148 formas de escribir la unidad de venta, agrupadas en 15 tipos de envase.


,formas,filas
tipo de envase,,
caja,56,43.522
bandeja,22,39.357
malla,14,16.143
unidad,1,13.733
kilo,10,11.172
saco,7,9.192
bins,4,8.891
docena,10,6.816
paquete,11,4.634


**Anomalía 6 · La unidad de venta se escribe de 148 formas.** Cada forma combina un envase (bandeja, caja, malla, saco, bins, atado, docena…) con un peso o una cantidad, y a veces con una palabra más, como `granel` o `empedrada`. Mientras no sepamos qué trae cada envase, los precios de un mismo producto no se pueden comparar. Por eso cada forma se revisó a mano y quedó anotada en una lista, `etl/reviewed_units.csv`. Si aparece una forma nueva, la carga se detiene hasta que alguien la revise. Hoy todas las de 2026 están en la lista.

Con la lista revisada podemos agregar a cada fila cuánto trae su envase, en kilos o en unidades. Las formas que la lista deja sin lectura son las que no permiten calcular un precio por kilo.

In [15]:
# La lista revisada a mano dice cuánto trae cada forma; las que no tienen lectura segura quedan vacías.
reviewed = pd.read_csv("../etl/reviewed_units.csv").rename(columns={"unit": UNIT})
raw = raw.merge(reviewed, on=UNIT, how="left")
no_reading = raw[raw["expected_unit"].isna()]
print(
    f"Se cruzaron las {number(raw[UNIT].nunique())} formas con la lista revisada: "
    f"{number((~raw[UNIT].isin(reviewed[UNIT])).sum())} no figuran en ella."
)
print(
    f"Se encontraron {number(len(no_reading))} filas, en {no_reading[UNIT].nunique()} formas, sin una "
    "lectura segura de su contenido."
)
show(no_reading[UNIT].value_counts().head(8).rename_axis("unidad").to_frame("filas"))

Se cruzaron las 148 formas con la lista revisada: 0 no figuran en ella.
Se encontraron 8.276 filas, en 15 formas, sin una lectura segura de su contenido.


,filas
unidad,
$/docena de atados,2.465
$/docena de matas,2.120
$/caja 36 atados,1.137
"$/atado 0,5 a 1 kilo",693
$/docena de paquetes,664
$/paquete,324
"$/atado 1,5 a 2 kilos",209
$/atado 300 a 500 gramos,189


**Anomalía 7 · Unidades sin un peso definido.** Hay 8.276 filas en esa situación. Algunas unidades no dicen cuánto pesan, como `docena de atados`, y otras dan solo un rango, como `atado 0,5 a 1 kilo`; un atado puede pesar medio kilo o uno entero. No inventamos un peso. Estas filas se publican sin precio por kilo y quedan marcadas, sin borrarse, porque es mejor un precio que falta que uno inventado.

Con la calidad pasa algo parecido. Listamos todas las calidades del archivo, con su número de filas, para ver si describen una sola escala.

In [16]:
# Todas las calidades publicadas, con su número de filas.
print(f"Se encontraron {raw['Calidad'].nunique()} calidades distintas.")
show(raw["Calidad"].value_counts().rename_axis("calidad").to_frame("filas"))

Se encontraron 55 calidades distintas.


,filas
calidad,
Primera,76.248
Segunda,27.045
Especial,12.543
Tercera,6.611
1a (cosecha),4.086
1a (guarda),4.023
Extra,2.901
1a amarillo,2.759
1a nueva(o),2.092


**Anomalía 8 · Las calidades no forman una escala.** Son 55, y mezclan criterios distintos: grado (`Primera`), calibre (`Calibre 80`), madurez (`Pintón`) y cosecha (`1a (guarda)`). Hasta la escritura cambia: `1a nueva(o)` va sin espacio y `3a nueva (o)`, con espacio. El archivo no dice si `Especial` está por encima de `Extra`, ni cómo se compara un calibre con un grado. Dejamos las calidades tal como las escribe ODEPA y comparamos siempre dentro de una misma calidad, sin ordenarlas de mejor a peor.

En la variedad, el valor más frecuente no es una variedad. Medimos cuántas filas lo llevan y en qué productos.

In [17]:
# La etiqueta más frecuente de la variedad y los productos que más la llevan.
top = raw["Variedad / Tipo"].value_counts().index[0]
unknown = raw[raw["Variedad / Tipo"] == top]
print(
    f"Se encontraron {number(len(unknown))} filas con la variedad «{top}» "
    f"({str(round(len(unknown) / len(raw) * 100, 1)).replace('.', ',')} % del archivo)."
)
show(unknown["Producto"].value_counts().head(5).rename_axis("producto").to_frame("filas"))

Se encontraron 56.606 filas con la variedad «Sin especificar» (36,3 % del archivo).


,filas
producto,
Limón,7.886
Cebolla,5.869
Frutilla,3.578
Zanahoria,2.917
Brócoli,2.860


**Anomalía 9 · Variedad «Sin especificar».** El 36,3 % de las filas, 56.606, no dice la variedad, y el limón es el producto con más filas así. Esa etiqueta puede juntar variedades con precios distintos. La tratamos como una variedad más; no la repartimos ni la completamos, porque el archivo no permite saber cuál era.

Queda el origen. Clasificamos cada valor según su nivel, región, provincia u otro, y miramos los que no encajan.

In [18]:
# El nivel de cada origen: región, provincia u otro.
level = raw["Origen"].str.split().str[0].where(raw["Origen"].str.startswith(("Provincia", "Región")), "Otro")
print(
    f"Se encontraron {number(int((level == 'Región').sum()))} orígenes a nivel de región, "
    f"{number(int((level == 'Provincia').sum()))} a nivel de provincia y {number(int((level == 'Otro').sum()))} "
    "de otro nivel."
)
show(raw.loc[level == "Otro", "Origen"].value_counts().head(7).rename_axis("origen").to_frame("filas"))

Se encontraron 83.346 orígenes a nivel de región, 53.411 a nivel de provincia y 19.100 de otro nivel.


,filas
origen,
Perú,6.798
Ecuador,6.763
China,2.266
Brasil,1.461
Cabildo,673
Limache,441
Paine,144


**Anomalía 10 · El origen mezcla niveles.** Hay regiones, provincias, comunas como `Cabildo` o `Limache`, y países como `Perú`, `Ecuador` o `Brasil`. Algunos niveles se pisan, como `Región Metropolitana` y `Provincia de Santiago`, así que no se puede sumar por origen sin elegir uno. Como el análisis es por región de venta y no de producción, el origen no entra en ningún cálculo. Lo guardamos en la capa cruda, por si algún día se necesita.

**Paso 6 · ¿Un mismo producto tiene precios de escalas distintas?**

Este es el paso más largo, porque aquí se decide cómo se lee cada unidad. La idea es sencilla: si una lectura es correcta, el precio por kilo de un producto tiene que parecerse en todos sus envases. Empezamos por ver qué unidades tienen los precios más altos.

In [19]:
# Precio mediano de cada unidad: las más caras primero.
scale = raw.groupby(UNIT)["avg"].agg(rows="size", median_price="median").sort_values("median_price", ascending=False)
print(
    f"El precio mediano de una forma típica es de ${number(scale['median_price'].median())}; "
    "estas son las seis más caras."
)
show(
    scale.head(6).rename(columns={"rows": "filas", "median_price": "precio_mediano"}).rename_axis("unidad"),
    mark=["precio_mediano"],
)

El precio mediano de una forma típica es de $12.000; estas son las seis más caras.


,filas,precio_mediano
unidad,,
$/bins (420 kilos),684,305.278
$/bins (450 kilos),2.271,275.000
$/bins (400 kilos),5.934,250.000
$/bins (350 kilos),2,195.000
$/caja 30 kilos,2,65.000
$/malla 50 unidades,22,42.500


Los bins están muy por encima del resto. Para entender por qué, tomamos el producto con más filas en bins y llevamos el precio de cada uno de sus envases a kilo, usando el peso que dice el texto.

In [20]:
# El peso que dice el texto de cada unidad (por ejemplo, 400 en «$/bins (400 kilos)»).
def weight(unit):
    found = re.search(r"(\d+(?:,\d+)?) kilos", unit)
    return float(found.group(1).replace(",", ".")) if found else float("nan")


raw["weight"] = raw[UNIT].map({unit: weight(unit) for unit in raw[UNIT].unique()})
packaged = raw[UNIT].str.contains(r"^\$/(?:(?:bandeja|caja|malla|saco) \d|bins \(\d)")
bins = raw[raw[UNIT].str.contains("bins")]
product = bins["Producto"].value_counts().index[0]
detail = (
    raw[(raw["Producto"] == product) & packaged]
    .groupby(UNIT)
    .agg(rows=("avg", "size"), median_price=("avg", "median"), kilos=("weight", "first"))
)
detail["price_per_kilo"] = detail["median_price"] / detail["kilos"]
print(f"El producto con más filas en bins es {product}; estos son sus envases con al menos 30 filas.")
display(
    show(
        detail[detail["rows"] >= 30]
        .sort_values("median_price")
        .rename(
            columns={
                "rows": "filas",
                "median_price": "precio_mediano",
                "kilos": "kilos",
                "price_per_kilo": "precio_por_kilo",
            }
        )
        .rename_axis("unidad"),
        mark=["precio_por_kilo"],
    )
)
show(bins[bins["Producto"] == product].head(3)[["Fecha", UNIT, "Volumen", "Precio promedio"]])

El producto con más filas en bins es Manzana; estos son sus envases con al menos 30 filas.


,filas,precio_mediano,kilos,precio_por_kilo
unidad,,,,
$/bandeja 15 kilos granel,3.640,12.000,15,800
$/caja 16 kilos empedrada,432,12.000,16,750
$/caja 18 kilos granel,271,14.000,18,778
$/bandeja 18 kilos granel,698,15.000,18,833
$/caja 15 kilos empedrada,596,17.500,15,1.167
$/caja 18 kilos embalada,1.162,23.000,18,1.278
$/caja 20 kilos granel,42,25.300,20,1.265
$/bins (400 kilos),3.180,260.000,400,650


Fecha,Unidad de comercializacion,Volumen,Precio promedio
2026-01-14,$/bins (400 kilos),10,"400000,0000"
2026-01-15,$/bins (400 kilos),20,"375000,0000"
2026-01-16,$/bins (400 kilos),10,"350000,0000"


**Anomalía 11 · El precio del bin es por un bin.** Los bins cuestan entre $250.000 y $305.278, mientras las bandejas y cajas de la manzana, el producto con más filas en bins, cuestan entre $12.000 y $25.300. Por kilo, en cambio, la manzana vale parecido en todos sus envases: $650 en el bin de 400 kilos y entre $750 y $1.278 en bandejas y cajas. La diferencia de precio se explica porque un bin trae 400 kilos. Queda una duda: una fila dice volumen 10 y precio $400.000, ¿es el precio de un bin o de los diez? Si fuera de los diez, el kilo costaría $100, entre siete y trece veces menos que en bandeja, y eso no calza con nada. El precio es de un solo bin, y lo dividimos por los 400 kilos que trae.

Algunas unidades dicen «$/kilo» y a la vez nombran una caja, como «$/kilo (en caja de 17 kilos)». ¿El precio es de un kilo o de la caja? Probamos las dos lecturas contra el precio por kilo del mismo producto en bandejas, cajas, mallas y sacos. La correcta es la que queda cerca de 1.

In [21]:
# Precio por kilo de referencia de cada producto: sus bandejas, cajas, mallas, sacos y bins con peso en el texto.
reference = (raw.loc[packaged, "avg"] / raw.loc[packaged, "weight"]).groupby(raw.loc[packaged, "Producto"]).median()
inside = raw[raw[UNIT].str.startswith("$/kilo (en ")]
readings = (
    inside.groupby(["Producto", UNIT])
    .agg(rows=("avg", "size"), published=("avg", "median"), kilos=("weight", "first"))
    .reset_index()
    .join(reference.rename("reference"), on="Producto")
)
readings["si_es_por_kilo"] = readings["published"] / readings["reference"]
readings["si_es_por_caja"] = readings["published"] / readings["kilos"] / readings["reference"]
print("Precio publicado frente a la referencia por kilo de cada producto, bajo las dos lecturas posibles.")
show(
    readings[readings["rows"] >= 30].rename(
        columns={"rows": "filas", "published": "publicado", "reference": "referencia"}
    ),
    mark=["si_es_por_kilo", "si_es_por_caja"],
)

Precio publicado frente a la referencia por kilo de cada producto, bajo las dos lecturas posibles.


Producto,Unidad de comercializacion,filas,publicado,kilos,referencia,si_es_por_kilo,si_es_por_caja
Chirimoya,$/kilo (en caja de 15 kilos),139,3.160,15,2.250,"1,4","0,09"
Naranja,$/kilo (en caja de 20 kilos),76,950,20,533,"1,78","0,09"
Palta,$/kilo (en bandeja de 18 kilos),706,3.500,18,3.065,"1,14","0,06"
Palta,$/kilo (en caja de 15 kilos),588,3.000,15,3.065,"0,98","0,07"
Palta,$/kilo (en caja de 17 kilos),6.539,2.800,17,3.065,"0,91","0,05"


**Anomalía 12 · «$/kilo (en caja de 17 kilos)» es precio por kilo.** Leído como precio por kilo, queda entre 0,91 y 1,78 veces la referencia. Leído como precio de la caja, cae a entre 0,05 y 0,09. En la palta de la caja de 17 kilos es 0,91 contra 0,05. Solo la lectura por kilo tiene sentido; la caja indica cuánto pesa cada envase que se cuenta en el volumen. Es una inferencia nuestra: los datos la respaldan, aunque ODEPA no lo declara.

La columna «Volumen» tampoco dice en qué se mide. Si fueran kilos, una fila de bins de 400 kilos debería marcar al menos 400. Medimos, para cada forma con peso, qué porcentaje de sus filas tiene un volumen menor que el peso de su envase.

In [22]:
# Porcentaje de filas cuyo volumen es menor que el peso del envase.
weighted = raw[packaged]
below = (
    weighted.assign(below=weighted["volume"] < weighted["weight"])
    .groupby(UNIT)
    .agg(rows=("below", "size"), pct=("below", "mean"))
)
below["pct"] = (below["pct"] * 100).round()
bins_rows = weighted[weighted[UNIT].str.contains("bins")]
print(
    f"Se encontraron {number(len(bins_rows))} filas en bins; en "
    f"{number((bins_rows.volume < bins_rows.weight).sum())} el volumen es menor que el peso del envase."
)
print("Porcentaje de filas con un volumen menor que el peso del envase, por forma.")
show(
    below[below["rows"] >= 100]
    .sort_values("pct", ascending=False)
    .head(6)
    .rename(columns={"rows": "filas", "pct": "% con volumen menor"})
    .rename_axis("unidad"),
    mark=["% con volumen menor"],
)

Se encontraron 8.891 filas en bins; en 8.891 el volumen es menor que el peso del envase.
Porcentaje de filas con un volumen menor que el peso del envase, por forma.


,filas,% con volumen menor
unidad,,
$/bins (450 kilos),2.271,100
$/bins (420 kilos),684,100
$/bins (400 kilos),5.934,100
$/caja 25 kilos,498,24
$/caja 18 kilos,4.076,1
$/caja 24 kilos,508,1


**Anomalía 13 · El volumen se cuenta en envases.** En los bins, las 8.891 filas tienen un volumen menor que el peso del envase. Ninguna otra forma se acerca: la que más, la caja de 25 kilos, llega a 24 %. Si el volumen fueran kilos, ninguna de esas filas alcanzaría a llenar un bin. Lo leemos como número de envases, salvo en las formas que dicen «(volumen en unidades)», y el precio de la semana se pondera por lo que trae cada envase. También es una inferencia: coincide con los datos y con el boletín de ODEPA, que expresa el volumen por envase, pero el archivo no lo documenta.

Cuatro formas avisan en su propio texto que el volumen va «en unidades». Las listamos con sus productos y vemos si el volumen es múltiplo del tamaño del paquete.

In [23]:
# Tamaño del paquete que dice el texto: 100 para «cien», el número antes de «unidades», o 1.
def pack_size(unit):
    if "cien" in unit:
        return 100
    found = re.search(r"(\d+) unidades", unit)
    return int(found.group(1)) if found else 1


flagged = raw[raw[UNIT].str.contains("volumen")]
flagged = flagged.assign(multiple=flagged["volume"] % flagged[UNIT].map(pack_size) == 0)
by_unit = flagged.groupby(UNIT).agg(
    rows=("avg", "size"),
    products=("Producto", lambda s: ", ".join(s.value_counts().index[:2])),
    pct_multiple=("multiple", "mean"),
)
by_unit["pct_multiple"] = (by_unit["pct_multiple"] * 100).round()
print(f"Se encontraron {len(by_unit)} formas que indican «(volumen en unidades)».")
show(
    by_unit.rename(
        columns={"rows": "filas", "products": "productos", "pct_multiple": "% múltiplo del paquete"}
    ).rename_axis("unidad")
)

Se encontraron 4 formas que indican «(volumen en unidades)».


,filas,productos,% múltiplo del paquete
unidad,,,
$/cien unidades (volumen en unidades),148,Rabanito,100
$/kilo (volumen en unidades),2.599,"Zapallo, Sandia",100
$/paquete 10 unidades (volumen en unidades),168,Cebolla,100
$/paquete 20 unidades (volumen en unidades),730,"Cebolla, Ajo",99


**Anomalía 14 · Precio por kilo, volumen en piezas.** La forma principal es «$/kilo (volumen en unidades)», con 2.599 filas de zapallo y sandía. El precio es por kilo, pero el volumen cuenta piezas enteras, y un zapallo no pesa siempre lo mismo. En las otras tres formas el volumen sí es múltiplo del tamaño del paquete. Hoy cada pieza se pondera como si pesara un kilo. Decidimos mantenerlo y dejarlo escrito en la regla, porque no cambia el precio de cada día, solo cuánto pesa ese día dentro de la semana, e inventar un peso por pieza sería peor.

Falta «cien». Calculamos el precio por unidad con las dos lecturas posibles y lo comparamos con lo que cuesta, en el mismo subsector, lo que se vende por unidad.

In [24]:
# El precio por unidad de «cien» con las dos lecturas, frente a lo que se vende por unidad en su subsector.
hundred = raw[raw[UNIT].str.contains("cien")]
subsector = hundred["Subsector"].iloc[0]
per_unit = raw[(raw[UNIT] == "$/unidad") & (raw["Subsector"] == subsector)]["avg"]
print("Precio por unidad de «cien» bajo cada lectura, junto a referencias del subsector.")
show(
    pd.DataFrame(
        {
            "precio por unidad": [
                hundred["avg"].median(),
                hundred["avg"].median() / 100,
                per_unit.median(),
                per_unit.quantile(0.9),
            ]
        },
        index=[
            f"{hundred['Producto'].iloc[0]}, si «cien» es una unidad",
            f"{hundred['Producto'].iloc[0]}, si «cien» son 100 unidades",
            "referencia: mediana de lo vendido por unidad",
            "referencia: percentil 90",
        ],
    ).rename_axis("lectura")
)

Precio por unidad de «cien» bajo cada lectura, junto a referencias del subsector.


,precio por unidad
lectura,
"Rabanito, si «cien» es una unidad",3.500
"Rabanito, si «cien» son 100 unidades",35
referencia: mediana de lo vendido por unidad,1.000
referencia: percentil 90,2.000


**Anomalía 15 · «Cien» son cien unidades.** La forma tiene 148 filas, todas de rabanito, con una mediana de $3.500. Si eso fuera el precio de un rabanito, costaría más que el 90 % de lo que se vende por unidad en su subsector, cuyo percentil 90 es $2.000. Leído como cien unidades, un rabanito cuesta $35, que es razonable. Dividimos el precio por 100. Como la 12 y la 13, es una inferencia que respaldan los datos, aunque ODEPA no la declara.

**Paso 7 · ¿Hay precios fuera de lo habitual?**

Un precio de otra escala suele ser un error de digitación o un envase mal anotado, y basta uno para mover el promedio de la semana. Comparamos cada precio con la mediana de su grupo (mismo producto, variedad, calidad, unidad y mercado) y listamos los que se alejan más de cinco veces.

In [25]:
# Cuántas veces es cada precio la mediana de su grupo; solo grupos de al menos 10 filas.
group = raw.groupby(["Producto", "Variedad / Tipo", "Calidad", UNIT, "Mercado"])["avg"]
raw["group_median"] = group.transform("median")
raw["times"] = raw["avg"] / raw["group_median"]
enough = raw[group.transform("size") >= 10]
far = enough[(enough["times"] > 5) | (enough["times"] < 0.2)]
print(
    f"Se encontraron {len(far)} filas que se alejan más de cinco veces de la mediana de su grupo; "
    "estas son las tres más bajas y las tres más altas."
)
show(
    pd.concat([far.nsmallest(3, "times"), far.nlargest(3, "times")])[
        ["Fecha", "Mercado", "Producto", UNIT, "Precio promedio", "group_median", "times"]
    ].rename(columns={"group_median": "mediana del grupo", "times": "veces la mediana"}),
    mark=["veces la mediana"],
)

Se encontraron 96 filas que se alejan más de cinco veces de la mediana de su grupo; estas son las tres más bajas y las tres más altas.


Fecha,Mercado,Producto,Unidad de comercializacion,Precio promedio,mediana del grupo,veces la mediana
2026-05-08,Vega Central Mapocho de Santiago,Apio,$/docena de matas,"600,0000",9.000,"0,07"
2026-03-11,Mercado Mayorista Lo Valledor de Santiago,Cebolla,$/caja 20 kilos,"500,0000",6.000,"0,08"
2026-01-07,Mercado Mayorista Lo Valledor de Santiago,Poroto verde,$/saco 25 kilos,"2000,0000",22.000,"0,09"
2026-03-19,Vega Modelo de Temuco,Pera,$/bandeja 18 kilos granel,"320000,0000",14.250,"22,46"
2026-03-11,Vega Monumental Concepción,Coliflor,$/unidad,"10000,0000",800,"12,5"
2026-07-20,Vega Modelo de Temuco,Sandia,$/kilo (volumen en unidades),"15000,0000",1.400,"10,71"


**Anomalía 16 · Precios fuera de escala.** Aparecen 96 filas. La más extrema es una pera en bandeja de 18 kilos publicada a $320.000, cuando esa bandeja suele costar $14.250, la mediana de su grupo; la cifra se parece más al precio de un bin. En el otro extremo, una caja de cebolla de 20 kilos aparece a $500. El archivo no permite saber si son errores de digitación, envases mal registrados o precios reales, así que no corregimos ningún número. Las filas se marcan como atípicas (la aplicación las compara con el resto de su misma semana) y quedan fuera de los indicadores, sin borrarse. Esta pera, por ejemplo, figura marcada en la base.

**Paso 8 · ¿Cambian los envases de una semana a otra?**

El último paso mira el tiempo. Si un producto se vende una semana en bandeja de 18 kilos y la siguiente en bandeja de 10, puede parecer que su precio subió sin que nada haya cambiado. Para cada producto, variedad, calidad y región comparamos los envases de semanas consecutivas y buscamos los casos en que no comparten ninguno.

In [26]:
# Envases y precio por kilo de cada semana, y los de la semana anterior de la misma serie.
raw["per_kilo"] = (raw["avg"] / raw["weight"]).where(packaged)
raw["week"] = pd.to_datetime(raw["Fecha"]).dt.to_period("W-SUN").dt.start_time
series = ["Producto", "Variedad / Tipo", "Calidad", "Region"]
weekly = (
    raw.groupby(series + ["week"])
    .agg(units=(UNIT, lambda s: " · ".join(sorted(set(s)))), price=("per_kilo", "mean"))
    .reset_index()
    .sort_values(series + ["week"])
)
previous = weekly.groupby(series)
weekly["units_before"], weekly["price_before"] = previous["units"].shift(), previous["price"].shift()
weekly["week_before"] = previous["week"].shift()
consecutive = (weekly["week"] - weekly["week_before"]).dt.days == 7
shared = pd.Series(
    [
        len(set(a.split(" · ")) & set(b.split(" · "))) if isinstance(b, str) else -1
        for a, b in zip(weekly["units"], weekly["units_before"])
    ],
    index=weekly.index,
)
cases = weekly[consecutive & (shared == 0)].copy()
cases["jump"] = (cases["price"] / cases["price_before"] - 1) * 100
latest = cases[cases["week"] == cases["week"].max()].sort_values("jump", ascending=False)
print(
    f"Se compararon {number(consecutive.sum())} pares de semanas consecutivas; {number(len(cases))} no "
    f"comparten ningún envase y {len(latest)} corresponden a la última semana."
)
show(
    latest.head(5)[series + ["units_before", "units", "price_before", "price", "jump"]].rename(
        columns={
            "units_before": "envase antes",
            "units": "envase ahora",
            "price_before": "precio por kilo antes",
            "price": "precio por kilo ahora",
            "jump": "salto aparente (%)",
        }
    ),
    mark=["salto aparente (%)"],
)

Se compararon 32.524 pares de semanas consecutivas; 827 no comparten ningún envase y 20 corresponden a la última semana.


Producto,Variedad / Tipo,Calidad,Region,envase antes,envase ahora,precio por kilo antes,precio por kilo ahora,salto aparente (%)
Kiwi,Hayward,Especial,Región del Maule,$/bandeja 18 kilos,$/bandeja 10 kilos,1.111,1.900,71
Naranja,Navel Late,Especial,Región de La Araucanía,$/bandeja 15 kilos granel,$/caja 15 kilos granel,1.075,1.333,24
Manzana,Pink Lady,Primera,Región del Biobío,$/bandeja 15 kilos granel,$/caja 16 kilos empedrada,867,938,"8,17"
Cebolla,Sin especificar,1a (guarda),Región de La Araucanía,$/malla 15 kilos,$/malla 18 kilos,1.031,1.111,"7,76"
Manzana,Granny Smith,Especial,Región del Biobío,$/bandeja 15 kilos granel,$/caja 16 kilos empedrada,1.067,1.109,4


**Anomalía 17 · Cambios de envase que parecen cambios de precio.** De 32.524 pares de semanas consecutivas, 827 no comparten ningún envase. En la última semana hay 20, y el que más salta es el kiwi Hayward Especial del Maule: pasa de bandeja de 18 kilos, a $1.111 el kilo, a bandeja de 10, a $1.900. Parece que subió 71 %, pero lo que cambió fue el envase. Por eso toda variación se calcula solo entre precios del mismo envase en las dos semanas, y si no hay uno en común, ese precio queda fuera de la comparación. Con esta regla, en la aplicación la alerta de este kiwi bajó de +25,6 % a +8,8 % y dejó de ser alerta.

**En resumen**

El procedimiento encontró 17 anomalías. Esta tabla reúne lo que hace la aplicación con cada una.

| # | Anomalía | Tratamiento en la aplicación | Estado |
|---|---|---|---|
| 1 | Números publicados como texto, con coma decimal | Convierte a número y conserva el texto original | Tratada |
| 2 | Días y mercados sin datos | No rellena; una semana faltante detiene la carga | Tratada |
| 3 | Precio promedio distinto del punto medio | Usa el promedio publicado por ODEPA | Tratada |
| 4 | Mínimo y máximo muy distantes | Usa solo el precio promedio | Tratada |
| 5 | Filas repetidas que difieren en el origen | Conserva todas las filas | Tratada |
| 6 | Unidad de venta escrita de 148 formas | Lista revisada manualmente; una forma nueva detiene la carga | Tratada |
| 7 | Unidades sin peso definido | Publica sin precio por kilo, con marca, sin eliminar | Tratada |
| 8 | Calidades sin un orden definido | Compara dentro de una misma calidad | Tratada |
| 9 | Variedad «Sin especificar» | La trata como una variedad más | Tratada |
| 10 | Origen con niveles distintos | No lo usa en los cálculos | Tratada |
| 11 | Precio por envase y no por el volumen total | Divide el precio por el contenido del envase | Tratada |
| 12 | «$/kilo (en caja de 17 kilos)» | Lo trata como precio por kilo | Tratada · inferencia |
| 13 | Volumen expresado en envases | Pondera la semana por el contenido de cada envase | Tratada · inferencia |
| 14 | «$/kilo (volumen en unidades)» | Pondera cada pieza como 1 kilo | Tratada |
| 15 | «$/cien unidades» | Lo trata como 100 unidades | Tratada · inferencia |
| 16 | Precios fuera de escala | Marca como atípicas, sin eliminar | Tratada |
| 17 | Cambio de envase entre semanas | Compara solo el mismo envase | Tratada |

Las 17 quedan tratadas. Tres de ellas, la 12, la 13 y la 15, se apoyan en inferencias: los datos las respaldan, aunque ODEPA no las declara.